In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishakhdapat/imdb-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/imdb-movie-reviews


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
import pandas as pd
df=pd.read_csv('/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
df['review'] = df['review'].str.replace('<br />', '', regex=False)

In [6]:
df['review']

0        One of the other reviewers has mentioned that ...
1        A wonderful little production. The filming tec...
2        I thought this was a wonderful way to spend ti...
3        Basically there's a family where a little boy ...
4        Petter Mattei's "Love in the Time of Money" is...
                               ...                        
49995    I thought this movie did a down right good job...
49996    Bad plot, bad dialogue, bad acting, idiotic di...
49997    I am a Catholic taught in parochial elementary...
49998    I'm going to have to disagree with the previou...
49999    No one expects the Star Trek movies to be high...
Name: review, Length: 50000, dtype: object

In [5]:
X = df['review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

# Tokenize reviews
tokenized_reviews = [word_tokenize(review.lower()) for review in X_train]

# Train Word2Vec model
w2v_model = Word2Vec(tokenized_reviews,
                    vector_size=300,  # embedding dimension
                    window=5,         # context window size
                    min_count=5,      # minimum word frequency
                    workers=4,        # parallel threads
                    epochs=10)       # training iterations

# Save the model
w2v_model.save("word2vec_imdb.model")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
import numpy as np

def average_word_vectors(review, model, vocabulary, num_features):
    feature_vector = np.zeros((num_features,), dtype="float64")
    nwords = 0
    
    for word in word_tokenize(review.lower()):
        if word in vocabulary:
            feature_vector = np.add(feature_vector, model.wv[word])
            nwords += 1
    
    if nwords:
        feature_vector = np.divide(feature_vector, nwords)
        
    return feature_vector

# Get vocabulary
vocabulary = set(w2v_model.wv.index_to_key)

# Create averaged feature vectors
X_train_w2v = np.array([average_word_vectors(review, w2v_model, vocabulary, 300) 
                       for review in X_train])
X_test_w2v = np.array([average_word_vectors(review, w2v_model, vocabulary, 300) 
                      for review in X_test])

In [8]:
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)

y_pred1 = lr_w2v.predict(X_test_w2v)
print("Logistic Regression with Word2Vec Accuracy:", accuracy_score(y_test, y_pred1))
print(classification_report(y_test, y_pred1))

Logistic Regression with Word2Vec Accuracy: 0.8726
              precision    recall  f1-score   support

    negative       0.88      0.87      0.87      4961
    positive       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [4]:
from transformers import BertTokenizer, BertModel
import torch

# Initialize BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [6]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

# Initialize BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
def get_bert_embeddings(texts, batch_size=32):
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Tokenize and prepare inputs
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get BERT outputs
        with torch.no_grad():
            outputs = bert_model(**inputs)
        
        # Use mean of last hidden states as document embedding
        embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.extend(embeddings)
    
    return np.array(all_embeddings)

# Generate BERT embeddings (this may take some time)
X_train_bert = get_bert_embeddings(X_train.tolist())
X_test_bert = get_bert_embeddings(X_test.tolist())

In [9]:
from sklearn.linear_model import LogisticRegression

lr_bert = LogisticRegression(max_iter=1000)
lr_bert.fit(X_train_bert, y_train)

y_pred = lr_bert.predict(X_test_bert)
print("Logistic Regression with BERT Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression with BERT Accuracy: 0.8893
              precision    recall  f1-score   support

    negative       0.89      0.89      0.89      4961
    positive       0.89      0.89      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [13]:
# Identify misclassified examples by the Word2Vec model
incorrect_w2v = (y_pred1 == y_test)  # Word2Vec classified
correct_bert = (y_pred != y_test)  # BERT misclassified

# Find indices where Word2Vec fails but BERT succeeds
indices = np.where(incorrect_w2v & correct_bert)[0]

def is_valid_review(text, min_words=50, max_words=250):
    return min_words <= len(text.split()) <= max_words 

filtered_indices = [i for i in indices if is_valid_review(X_test.iloc[i])]

for i in filtered_indices[2:3]:  
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"Word2Vec Prediction: {y_pred1[i]}")
    print(f"BERT Prediction: {y_pred[i]}")
    print("-" * 80)


Review: I've never really been sure whether I liked this documentary or not. It was shown on Channel 4 before a cut down version of Revelations, and is on the Revelations video tape before the uncut show. The documentary is basically friends of Bill saying how great he was for an hour with video clips of the show mixed in, a bit like a trailer for the film you're about to watch. It also features David Letterman grovelling like a worm for dumping Bill off the his show before he died, the reason? Bill made a joke about how Pro-Life people should picket funerals, and Letterman had Pro-life advertising. Anyway look out for the video as Revelations is Bill at his ranting best :)
Actual Label: positive
Word2Vec Prediction: positive
BERT Prediction: negative
--------------------------------------------------------------------------------


In [13]:
# Identify misclassified examples by the Word2Vec model
incorrect_w2v = (y_pred1 != y_test)  # Word2Vec misclassified
correct_bert = (y_pred == y_test)  # BERT correctly classified

# Find indices where Word2Vec fails but BERT succeeds
indices = np.where(incorrect_w2v & correct_bert)[0]

def is_valid_review(text, min_words=50, max_words=250):
    return min_words <= len(text.split()) <= max_words 

filtered_indices = [i for i in indices if is_valid_review(X_test.iloc[i])]

# Print up to 3 cases where BERT succeeds but Word2Vec fails
print("Examples where BERT succeeds but Word2Vec fails:\n")
for i in filtered_indices[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"Word2Vec Prediction: {y_pred1[i]}")
    print(f"BERT Prediction: {y_pred[i]}")
    print("-" * 80)


Examples where BERT succeeds but Word2Vec fails:

Review: I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok to have a huge fat man vs a strong man but I'm glad times have changed. It was a terrible main event just like every match Luger is in is terrible. Other matches on the card were Razor Ramon vs Ted Dibiase, Steiner Brothers vs Heavenly Bodies, Shawn Michaels vs Curt Hening, this was the event where Shawn named his big monster of a body guard Diesel, IRS vs 1-2-3 Kid, Bret Hart first takes on Doink then takes on Jerry Lawler and stuff with the Harts and Lawler was always very interesting, then Ludvig Borga destroyed Marty Jannetty, Undertaker took on Giant Gonzalez in another terrible match, The Smoking Gunns and Tatanka took on Bam Ba

In [12]:
# Identify misclassified examples by the BERT model
incorrect_bert = (y_pred != y_test)  # BERT misclassified

# Find indices where BERT fails
indices = np.where(incorrect_bert)[0]

def is_valid_review(text, min_words=50, max_words=450):
    return min_words <= len(text.split()) <= max_words 


filtered_indices = [i for i in indices if is_valid_review(X_test.iloc[i])]

# Print up to 3 cases where BERT fails
print("Examples where BERT fails:\n")
for i in filtered_indices[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"BERT Prediction: {y_pred[i]}")
    print("-" * 80)


Examples where BERT fails:

Review: I was first introduced to John Waters films by seeing "Female trouble" on IFC. I was disgusted but for some sick reason i enjoyed it. Then, i picked up the Pink Flamingos DVD in the John Waters Boxed Set. The movie is about Babs Johnson "The Filthiest Person Alive" who lives in a trailer in Maryland with her obese egg obsessed mother,and her deranged son "Crackers". In the movie you will see such sick sights as sex with chickens, drag-queens, people eating feces, torture, and all other sorts of random humiliation. The film has a soundtrack from 60's rock and roll artists. The only problem is that some parts of the film seem to drag on and can get a little boring. I found "Female Trouble" a little more fun. Rated NC-17 for Explicit sex, violence, and disturbing images. Enjoy.
Actual Label: positive
BERT Prediction: negative
--------------------------------------------------------------------------------
Review: This movie has very good acting by virtu

In [9]:

# Convert to DataFrame
results_df = pd.DataFrame({'Text': X_test, 'True_Label': y_test, 'Predicted_Label': y_pred})

# Filter misclassified cases
false_negatives = results_df[(results_df['True_Label'] == 'positive') & (results_df['Predicted_Label'] == 'negative')]
false_positives = results_df[(results_df['True_Label'] == 'negative') & (results_df['Predicted_Label'] == 'positive')]

# Save to CSV files
false_negatives.to_csv('false_negatives.csv', index=False)
false_positives.to_csv('false_positives.csv', index=False)

print("Misclassified cases saved to CSV files.")

Misclassified cases saved to CSV files.


In [7]:
svm_bert = SVC(kernel='linear', probability=True)
svm_bert.fit(X_train_bert, y_train)

# Predict with SVM
y_pred_svm = svm_bert.predict(X_test_bert)
print("SVM with BERT Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM with BERT Accuracy: 0.8862
              precision    recall  f1-score   support

    negative       0.88      0.89      0.89      4961
    positive       0.89      0.88      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [16]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Sample original string labels (replace with your actual string labels)
# Example: y_train = ['positive', 'negative', 'positive', ...]
# If yours are already string labels, skip this example
# y_train and y_test must be original string labels like 'positive', 'negative'

# Step 1: Encode string labels to integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)   # e.g. 'negative' → 0, 'positive' → 1
y_test_encoded = label_encoder.transform(y_test)

# Step 2: Convert inputs and labels to NumPy arrays
X_train_bert = np.array(X_train_bert)
X_test_bert = np.array(X_test_bert)
y_train_array = np.array(y_train_encoded).reshape(-1, 1).astype(np.float32)
y_test_array = np.array(y_test_encoded).reshape(-1, 1).astype(np.float32)

# Step 3: Build the model
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_bert.shape[1],)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")  # Binary classification
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Step 4: Early stopping to avoid overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Step 5: Train the model
model.fit(
    X_train_bert, y_train_array,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_bert, y_test_array),
    callbacks=[early_stop]
)

# Step 6: Predictions and Evaluation
y_pred_probs = model.predict(X_test_bert)
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()
y_test_flat = y_test_array.flatten().astype(int)

# Step 7: Accuracy & Classification Report
print(f"\nAccuracy: {accuracy_score(y_test_flat, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_flat, y_pred))


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_label.py:134: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8397 - loss: 0.3650 - val_accuracy: 0.8419 - val_loss: 0.3535
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8791 - loss: 0.2881 - val_accuracy: 0.8754 - val_loss: 0.2935
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8837 - loss: 0.2750 - val_accuracy: 0.8882 - val_loss: 0.2657
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8884 - loss: 0.2687 - val_accuracy: 0.8892 - val_loss: 0.2650
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8902 - loss: 0.2608 - val_accuracy: 0.8853 - val_loss: 0.2705
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Accuracy: 0.8892

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      4961
           1       0.89      0.89      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000


SBERT

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
import numpy as np

X = df['review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load a larger SBERT model
sbert_model = SentenceTransformer('all-mpnet-base-v2')

# Convert text to SBERT embeddings
X_train_sbert = np.array(sbert_model.encode(X_train.tolist(), convert_to_numpy=True))
X_test_sbert = np.array(sbert_model.encode(X_test.tolist(), convert_to_numpy=True))

# Standardize features
scaler = StandardScaler()
X_train_sbert = scaler.fit_transform(X_train_sbert)
X_test_sbert = scaler.transform(X_test_sbert)

# Train Logistic Regression model
lr_sbert = LogisticRegression(max_iter=1000)
lr_sbert.fit(X_train_sbert, y_train)  

# Predictions
y_pred_sbert = lr_sbert.predict(X_test_sbert)

# Evaluation
print("Logistic Regression with SBERT Accuracy:", accuracy_score(y_test, y_pred_sbert))
print(classification_report(y_test, y_pred_sbert))


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Logistic Regression with SBERT Accuracy: 0.9084
              precision    recall  f1-score   support

    negative       0.92      0.90      0.91      4961
    positive       0.90      0.92      0.91      5039

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [14]:
# Identify indices where BERT fails but SBERT succeeds
failure_indices = np.where((y_pred != y_test) & (y_pred_sbert == y_test))[0]

# Extract reviews that meet the length condition
selected_reviews = []
for idx in failure_indices:
    review = X_test.iloc[idx]  
    if len(review.split()) <= 300:  
        selected_reviews.append((review, y_test.iloc[idx], y_pred[idx], y_pred_sbert[idx]))
    
    if len(selected_reviews) == 3:  
        break

# Print results
for i, (review, actual, bert_pred, sbert_pred) in enumerate(selected_reviews, 1):
    print(f"Case {i}:\n")
    print(f"Review: {review}\n")
    print(f"Actual Sentiment: {actual}")
    print(f"BERT Prediction: {bert_pred} ")
    print(f"SBERT Prediction: {sbert_pred} ")
    print("-" * 100)


Case 1:

Review: This movie has very good acting by virtually all the cast, a gripping story with a chilling ending, great music, and excellent visuals without significant special effects. It is interesting to note though that, like so much science fiction, its predictions for the future don't appear likely to come to pass as early as depicted. That's not to say we're out of the woods yet, but 2022 is now obviously too soon to be in this condition. It shares this failing with a fairly illustrious list of science fiction classics: "1984", "2001: A Space Odyssey (compare its space station with our International Space Station) and Isaac Asimov's "I Robot" (positronic brains were to have been invented in the 1990's).

Actual Sentiment: positive
BERT Prediction: negative 
SBERT Prediction: positive 
----------------------------------------------------------------------------------------------------
Case 2:

Review: I've never really been sure whether I liked this documentary or not. It was 

In [18]:
# Identify indices where SBERT fails (i.e., misclassification)
failure_indices_sbert = np.where(y_pred_sbert != y_test)[0]

# Extract reviews that meet the length condition
failed_reviews = []
for idx in failure_indices_sbert:
    review = X_test.iloc[idx]  
    if len(review.split()) <= 300:  
        failed_reviews.append((review, y_test.iloc[idx], y_pred_sbert[idx]))
    
    if len(failed_reviews) == 3:  # Stop once we have 3 cases
        break

# Print results
for i, (review, actual, sbert_pred) in enumerate(failed_reviews, 1):
    print(f"Case {i}:\n")
    print(f"Review: {review}\n")
    print(f"Actual Sentiment: {actual}")
    print(f"SBERT Prediction: {sbert_pred}")
    print("-" * 100)


Case 1:

Review: I was first introduced to John Waters films by seeing "Female trouble" on IFC. I was disgusted but for some sick reason i enjoyed it. Then, i picked up the Pink Flamingos DVD in the John Waters Boxed Set. The movie is about Babs Johnson "The Filthiest Person Alive" who lives in a trailer in Maryland with her obese egg obsessed mother,and her deranged son "Crackers". In the movie you will see such sick sights as sex with chickens, drag-queens, people eating feces, torture, and all other sorts of random humiliation. The film has a soundtrack from 60's rock and roll artists. The only problem is that some parts of the film seem to drag on and can get a little boring. I found "Female Trouble" a little more fun. Rated NC-17 for Explicit sex, violence, and disturbing images. Enjoy.

Actual Sentiment: positive
SBERT Prediction: negative
----------------------------------------------------------------------------------------------------
Case 2:

Review: A ruthless assassin has 

In [12]:
# Identify misclassified cases
misclassified_pos_as_neg = X_test[(y_test == 'positive') & (y_pred == 'negative')]
misclassified_neg_as_pos = X_test[(y_test == 'negative') & (y_pred == 'positive')]

# Save to CSV
pd.DataFrame(misclassified_pos_as_neg, columns=['text']).to_csv('misclassified_positive_as_negative.csv', index=False)
pd.DataFrame(misclassified_neg_as_pos, columns=['text']).to_csv('misclassified_negative_as_positive.csv', index=False)

In [ ]:
df.to_csv('imdb_sbert_results.csv', index=False)